# Classes, Objects, and Attribute Mechanics

**Session:** 2026-10-30 — OOP with Python  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

Object-Oriented Programming (OOP) is a paradigm centered on modeling software systems as interacting objects that encapsulate both **state** (attributes) and **behavior** (methods). This notebook explores the foundational mechanics of classes in Python: transitioning from procedural routines to encapsulated classes, understanding instance versus class attributes, mastering the `self` binding mechanism, exploring special dunder methods, and enforcing encapsulation with private attributes and `@property` descriptors.

## Learning Outcomes

- Contrast Procedural Programming (POP) with Object-Oriented Programming (OOP) through a concrete case study.
- Define classes, instantiate objects, and initialize state via `__init__`.
- Differentiate class attributes from instance attributes and understand attribute lookup order via `__dict__`.
- Explain the role of the `self` parameter in instance method dispatch.
- Implement callable instances using the `__call__` dunder protocol.
- Enforce encapsulation using public, protected (`_`), and private (`__`) naming conventions with name mangling.
- Use Pythonic `@property` getters and setters with data validation.

## 1. Paradigm Shift: Procedural vs Object-Oriented Modeling

In procedural programming, programs are organized into top-down sequences of functions that operate directly on shared or global data. As software scales, procedural architectures encounter significant friction:
- **Global state pollution:** Any function can mutate shared state, leading to unpredictable side-effects.
- **Tight coupling:** Data structures and algorithmic functions are disconnected, making changes in data schemas ripple through the codebase.
- **Low reusability:** Creating variants of existing logic (e.g., Chicago Style vs New York Style pizza) requires duplicated procedural branches.

Let us review the classic pizza preparation pipeline implemented procedurally:

In [ ]:
# Procedural implementation: reliance on global variables and discrete functions
pizza_name = ""
dough_type = ""
sauce_type = ""
toppings = []
bake_time_min = 0


def prepare_ny_style() -> None:
    global pizza_name, dough_type, sauce_type, toppings, bake_time_min
    pizza_name = "New York Style Thin Crust"
    dough_type = "Thin Crust Dough"
    sauce_type = "Marinara Sauce"
    toppings = ["Grated Reggiano Cheese", "Garlic"]
    bake_time_min = 25
    print(f"Prepared: {pizza_name} with {sauce_type} and {toppings}")


def bake_pizza() -> None:
    print(f"Baking {pizza_name} for {bake_time_min} minutes at 180°C.")


def slice_pizza(diagonal: bool = True) -> None:
    slice_style = "diagonal triangles" if diagonal else "square slices"
    print(f"Cutting {pizza_name} into {slice_style}.")


prepare_ny_style()
bake_pizza()
slice_pizza(diagonal=True)

Now let us refactor this into an Object-Oriented design. Each pizza becomes an independent instance with encapsulated attributes, and behaviors are self-contained methods:

In [ ]:
from collections.abc import Sequence


class Pizza:
    """Encapsulated representation of a pizza with its specific preparation routine."""

    def __init__(
        self,
        name: str,
        dough: str,
        sauce: str,
        toppings: Sequence[str],
        bake_time_minutes: int,
        slice_diagonal: bool = True,
    ) -> None:
        self.name = name
        self.dough = dough
        self.sauce = sauce
        self.toppings = list(toppings)
        self.bake_time_minutes = bake_time_minutes
        self.slice_diagonal = slice_diagonal

    def prepare(self) -> None:
        print(f"Preparing {self.name}: tossing {self.dough}, adding {self.sauce}...")
        print(f"Adding toppings: {', '.join(self.toppings)}")

    def bake(self) -> None:
        print(f"Baking {self.name} for {self.bake_time_minutes} minutes at 180°C.")

    def cut(self) -> None:
        style = "diagonal triangles" if self.slice_diagonal else "square slices"
        print(f"Cutting {self.name} into {style}.")

    def box(self) -> None:
        print(f"Placing {self.name} into official thermal delivery box.\n")


# Instantiating distinct pizza objects without global variable conflict
ny_pizza = Pizza(
    name="NY Style Sauce & Cheese",
    dough="Thin Crust",
    sauce="Marinara",
    toppings=["Grated Reggiano Cheese"],
    bake_time_minutes=25,
    slice_diagonal=True,
)

chicago_pizza = Pizza(
    name="Chicago Deep Dish",
    dough="Extra Thick Crust",
    sauce="Plum Tomato Sauce",
    toppings=["Shredded Mozzarella Cheese", "Italian Sausage"],
    bake_time_minutes=35,
    slice_diagonal=False,
)

for p in (ny_pizza, chicago_pizza):
    p.prepare()
    p.bake()
    p.cut()
    p.box()

## 2. Class Attributes vs Instance Attributes

In Python:
- **Class attributes** are defined directly within the class body. They are shared across all instances of the class and reside in `ClassName.__dict__`.
- **Instance attributes** are bound to a specific object instance (typically inside `__init__`) via `self.attribute_name` and reside in `instance.__dict__`.

When accessing `instance.attribute`, Python first checks `instance.__dict__`. If not found, it traverses upward to `Class.__dict__` and its base classes.

In [ ]:
class Dog:
    # Class attribute: shared constant across all dog instances
    species: str = "Canis familiaris"
    instance_count: int = 0

    def __init__(self, name: str, breed: str, age: int) -> None:
        # Instance attributes: unique to each dog instance
        self.name = name
        self.breed = breed
        self.age = age
        Dog.instance_count += 1


# Verify instantiation and namespace isolation
dog1 = Dog(name="Buddy", breed="Golden Retriever", age=3)
dog2 = Dog(name="Milo", breed="Beagle", age=2)

print(f"Dog 1: {dog1.name}, Species: {dog1.species}")
print(f"Dog 2: {dog2.name}, Species: {dog2.species}")
print(f"Total Dogs created: {Dog.instance_count}")

# Verify __dict__ contents
print(f"dog1 namespace: {dog1.__dict__}")
assert "name" in dog1.__dict__
assert "species" not in dog1.__dict__  # Species lives in Dog.__dict__
assert Dog.species == "Canis familiaris"
assert Dog.instance_count == 2

### Attribute Shadowing Pitfall

Assigning to `instance.class_attr` does **not** mutate the class attribute; rather, it creates a new instance attribute that **shadows** the class attribute on that specific object:

In [ ]:
# Shadowing demonstration
dog1.species = "Canis lupus"  # Creates instance attribute on dog1

print(f"dog1.species: {dog1.species} (shadowed)")
print(f"dog2.species: {dog2.species} (inherited from class)")
print(f"Dog.species:  {Dog.species} (class attribute unchanged)")

assert dog1.species == "Canis lupus"
assert dog2.species == "Canis familiaris"
assert Dog.species == "Canis familiaris"
assert "species" in dog1.__dict__

## 3. The `self` Parameter and Callable Instances (`__call__`)

In Python, instance methods require `self` as their first explicit argument:
- When you invoke `obj.method(arg)`, Python translates this internally to `Class.method(obj, arg)`.
- `self` binds the execution context to the specific calling instance.

Additionally, defining the dunder method `__call__` allows an instance of a class to be called directly like a function (`obj(...)`). This pattern is ubiquitous in machine learning (e.g., PyTorch modules, scikit-learn transformers).

In [ ]:
import math


class Point2D:
    """Geometric 2D point supporting vector operations and callable distance calculation."""

    def __init__(self, x: float = 0.0, y: float = 0.0) -> None:
        self.x = float(x)
        self.y = float(y)

    def distance_to(self, other: "Point2D") -> float:
        """Standard instance method."""
        return math.hypot(self.x - other.x, self.y - other.y)

    def translate(self, dx: float, dy: float) -> "Point2D":
        """Return a new translated Point2D."""
        return Point2D(self.x + dx, self.y + dy)

    def __call__(self, scale_factor: float) -> "Point2D":
        """Calling the point instance scales its coordinates from origin."""
        return Point2D(self.x * scale_factor, self.y * scale_factor)

    def __repr__(self) -> str:
        return f"Point2D(x={self.x}, y={self.y})"


p1 = Point2D(3.0, 4.0)
p2 = Point2D(0.0, 0.0)

# 1. Instance method call
dist = p1.distance_to(p2)
print(f"Distance from {p1} to {p2}: {dist:.2f}")
assert math.isclose(dist, 5.0)

# 2. Desugared method dispatch
assert Point2D.distance_to(p1, p2) == dist

# 3. Callable instance via __call__
scaled_p1 = p1(2.5)  # Invokes p1.__call__(2.5)
print(f"Scaled Point: {scaled_p1}")
assert scaled_p1.x == 7.5 and scaled_p1.y == 10.0

## 4. Encapsulation: Public, Protected, and Private Members

Python uses naming conventions rather than strict compiler-enforced access modifiers:
- **Public (`name`):** Accessible freely from anywhere.
- **Protected (`_name`):** Single leading underscore; indicates internal use by the class and subclasses. Not enforced by the interpreter, but respected by convention and linters.
- **Private (`__name`):** Double leading underscore without trailing underscore; triggers **name mangling**. The Python interpreter rewrites `__name` to `_ClassName__name` to prevent accidental name collisions in subclass hierarchies.

In [ ]:
class BankAccount:
    """Account demonstrating access levels and name mangling."""

    def __init__(self, account_holder: str, initial_balance: float) -> None:
        self.account_holder: str = account_holder  # Public
        self._routing_number: str = "021000021"  # Protected (by convention)
        self.__balance: float = float(initial_balance)  # Private (name mangled)

    def get_balance(self) -> float:
        return self.__balance

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Deposit amount must be positive.")
        self.__balance += amount


account = BankAccount("Ada Lovelace", 1000.0)

# Public access works directly
print(f"Holder: {account.account_holder}")

# Protected access is allowed but flagged as private by linters
print(f"Routing: {account._routing_number}")

# Direct access to private attribute raises AttributeError
try:
    print(account.__balance)
except AttributeError as err:
    print(f"Direct private access caught: {err}")

# Inspect name mangling in __dict__
print(f"Object namespace: {list(account.__dict__.keys())}")
assert "_BankAccount__balance" in account.__dict__
assert account._BankAccount__balance == 1000.0

## 5. Pythonic Encapsulation: The `@property` Decorator

Instead of Java-style explicit getters and setters (`get_balance()`, `set_balance()`), Python provides the `@property` decorator. This allows attribute access syntax (`account.balance`) while executing custom validation and invariant checks under the hood.

In [ ]:
class Student:
    """Student entity demonstrating @property getter, setter, and deleter."""

    def __init__(self, name: str, gpa: float) -> None:
        self.name = name
        self.gpa = gpa  # Routes through setter validation

    @property
    def name(self) -> str:
        return self._name

    @name.setter
    def name(self, value: str) -> None:
        if not isinstance(value, str) or not value.strip():
            raise ValueError("Student name must be a non-empty string.")
        self._name = value.strip()

    @property
    def gpa(self) -> float:
        return self._gpa

    @gpa.setter
    def gpa(self, value: float) -> None:
        if not (0.0 <= value <= 4.0):
            raise ValueError(f"GPA must be in range [0.0, 4.0], received {value}")
        self._gpa = float(value)

    def is_honor_roll(self) -> bool:
        return self._gpa >= 3.5


# Validation tests
student = Student(name="Grace Hopper", gpa=3.95)
print(f"Student: {student.name}, GPA: {student.gpa}, Honors: {student.is_honor_roll()}")

# Valid update
student.gpa = 4.0
assert student.gpa == 4.0

# Invariant violation rejected
try:
    student.gpa = 4.5
except ValueError as exc:
    print(f"Validation caught invalid GPA: {exc}")

## Summary

- **Procedural vs OOP:** Procedural organizes around actions and shared state; OOP organizes around autonomous entities combining state and actions.
- **Attributes:** Class attributes are shared; instance attributes are isolated in `self.__dict__`.
- **Method Dispatch:** `obj.method(...)` is syntactic sugar for `Class.method(obj, ...)`.
- **Access Modifiers:** Python employs naming conventions (`_protected`, `__private` with name mangling) rather than compiler keywords.
- **Properties:** Use `@property` for transparent, validated attribute interfaces.